### Import library

In [1]:
import os
os.chdir('../..')
os.getcwd()

'C:\\Users\\csia7\\OneDrive\\문서\\GitHub\\BJ46158_WQBrain_API'

In [2]:
import ace_lib as ace
import helpful_functions as hf
import pandas as pd
import requests
import plotly.express as px
import pygwalker as pyg
import glob
import re

### Start session
Enter credentials once - they will be saved to local folder and loaded each time

In [3]:
s = ace.start_session()

Complete biometrics authentication and press any key to continue: 
https://api.worldquantbrain.com/authentication/persona?inquiry=inq_4QBcLG6777Z1Ft8i3Ge9HDq5UUxJ

 


In [4]:
ace.check_session_timeout(s)

14399.10883

## Alpha Mixing Templates

In [5]:
'''
signal1 = scale_down(<cs_operator>(ts_backfill(<data1>,126)));

signal2 = scale_down(<cs_operator>(ts_backfill(<data2>,126)));

signal1 * (1 + signal2)
'''

'\nsignal1 = scale_down(<cs_operator>(ts_backfill(<data1>,126)));\n\nsignal2 = scale_down(<cs_operator>(ts_backfill(<data2>,126)));\n\nsignal1 * (1 + signal2)\n'

## Read all passed alphas

In [6]:
os.chdir('Results/signal')

In [7]:
file_direc = glob.glob('*')
file_direc

['A_signalform.csv',
 'B_signalform.csv',
 'C_signalform.csv',
 'D_signalform.csv',
 'pre_signalform.csv',
 'Sep03GLB_signalform.csv',
 'Sep13GLB_signalform.csv']

In [8]:
expressions = {}
for file in file_direc:
    expressions[file.split('_')[0]] = pd.read_csv(file)['expression']
#expressions

In [9]:
expressions.keys()

dict_keys(['A', 'B', 'C', 'D', 'pre', 'Sep03GLB', 'Sep13GLB'])

In [16]:
expressions['pre'][0]

'sig_pre_0 = group_zscore(ts_zscore(ts_backfill(anl69_epss_best_eeps_cur_yr, 21), 21), industry);factor_pre_0 = group_zscore(ts_ir(returns, 252), industry);value_pre_0 = floor(rank(fnd28_value_05480/close)*4.99);signal_pre_0 = group_neutralize(sig_pre_0, value_pre_0 *factor_pre_0)'

## Template Sep03GLB & Sep13GLB mixing
use top 32 results

In [11]:
pre = expressions['pre']
Sep13GLB = expressions['Sep13GLB']

In [12]:
print(len(pre), len(Sep13GLB))

1 30


In [23]:
Sep13GLB[0]

's1_Sep13GLB_0 = ts_zscore (ts_backfill (vec_avg(mdl139_score), 252), 252);s2_Sep13GLB_0 = ts_zscore (ts_backfill (vec_avg(oth193_shield2), 252), 252);group_Sep13GLB_0 = pv13_20_minvol_1m_sector;signal_Sep13GLB_0 = group_zscore(s1_Sep13GLB_0+s2_Sep13GLB_0, group_Sep13GLB_0)'

In [24]:
alpha_list_preSep13GLB = []

for j in range(15):
    alpha_list_preSep13GLB += [ace.generate_alpha(f'{pre[0]};{Sep13GLB[j]};add(scale_down(signal_pre_{0}), scale_down(signal_Sep13GLB_{j}), filter = false)', region= "GLB", universe = "MINVOL1M", neutralization = x, truncation = 0.01, decay = 3) for x in ['MARKET', 'COUNTRY']]

alpha_list_preSep13GLB

[{'type': 'REGULAR',
  'settings': {'instrumentType': 'EQUITY',
   'region': 'GLB',
   'universe': 'MINVOL1M',
   'delay': 1,
   'decay': 3,
   'neutralization': 'MARKET',
   'truncation': 0.01,
   'pasteurization': 'ON',
   'testPeriod': 'P0Y0M0D',
   'unitHandling': 'VERIFY',
   'nanHandling': 'OFF',
   'language': 'FASTEXPR',
   'visualization': False},
  'regular': 'sig_pre_0 = group_zscore(ts_zscore(ts_backfill(anl69_epss_best_eeps_cur_yr, 21), 21), industry);factor_pre_0 = group_zscore(ts_ir(returns, 252), industry);value_pre_0 = floor(rank(fnd28_value_05480/close)*4.99);signal_pre_0 = group_neutralize(sig_pre_0, value_pre_0 *factor_pre_0);s1_Sep13GLB_0 = ts_zscore (ts_backfill (vec_avg(mdl139_score), 252), 252);s2_Sep13GLB_0 = ts_zscore (ts_backfill (vec_avg(oth193_shield2), 252), 252);group_Sep13GLB_0 = pv13_20_minvol_1m_sector;signal_Sep13GLB_0 = group_zscore(s1_Sep13GLB_0+s2_Sep13GLB_0, group_Sep13GLB_0);add(scale_down(signal_pre_0), scale_down(signal_Sep13GLB_0), filter = fa

In [25]:
result = ace.simulate_alpha_list_multi(s, alpha_list_preSep13GLB)

 10%|████████▎                                                                          | 1/10 [00:04<00:37,  4.11s/it]

An error occurred
An error occurred
An error occurred


 40%|█████████████████████████████████▏                                                 | 4/10 [00:07<00:10,  1.73s/it]

An error occurred
An error occurred


 60%|█████████████████████████████████████████████████▊                                 | 6/10 [00:11<00:06,  1.74s/it]

An error occurred
An error occurred


 80%|██████████████████████████████████████████████████████████████████▍                | 8/10 [00:13<00:02,  1.46s/it]

An error occurred


100%|██████████████████████████████████████████████████████████████████████████████████| 10/10 [00:14<00:00,  1.48s/it]

An error occurred
An error occurred


In [14]:
#prettify_result function can be used from the helpful_functions library to take a look at IS stats of all the simulated alphas

result_st1 = hf.prettify_result(result, detailed_tests_view=False)
result_st1

,pnl,book_size,long_count,short_count,turnover,returns,drawdown,margin,fitness,sharpe,start_date,alpha_id,expression,concentrated_weight,high_turnover,is_ladder_sharpe,low_fitness,low_sharpe,low_sub_universe_sharpe,low_turnover
0,16886756,20000000,1646,1637,0.6236,0.1631,0.0202,0.000523,2.89,5.65,2012-01-22,K9jPPbp,"entry_B_0 = ts_rank( ts_std_dev(oth460_3l_vlc,...",PASS,PASS,PASS,PASS,PASS,PASS,PASS
1,17374738,20000000,1648,1635,0.6410,0.1678,0.0193,0.000523,2.89,5.65,2012-01-22,1dQYYgJ,"entry_B_0 = ts_rank( ts_std_dev(oth460_3l_vlc,...",PASS,PASS,PASS,PASS,PASS,PASS,PASS
2,16934881,20000000,1647,1636,0.6286,0.1635,0.0196,0.000520,2.87,5.62,2012-01-22,xk0Aw6m,"entry_B_0 = ts_rank( ts_std_dev(oth460_3l_vlc,...",PASS,PASS,PASS,PASS,PASS,PASS,PASS
3,17445468,20000000,1649,1634,0.6467,0.1685,0.0183,0.000521,2.87,5.62,2012-01-22,xk0AA2J,"entry_B_0 = ts_rank( ts_std_dev(oth460_3l_vlc,...",PASS,PASS,PASS,PASS,PASS,PASS,PASS
4,19869114,20000000,1556,1544,0.5654,0.1919,0.0372,0.000679,2.80,4.81,2012-01-22,k02q2Wz,entry_B_1 = ts_rank( ts_std_dev(oth460_1l_1dls...,PASS,PASS,PASS,PASS,PASS,PASS,PASS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,18730971,20000000,1625,1613,0.5785,0.1809,0.0331,0.000625,2.26,4.05,2012-01-22,Xn5mXeb,entry_B_8 = ts_rank( ts_std_dev(oth460_veia_l2...,PASS,PASS,PASS,PASS,PASS,PASS,PASS
96,19550126,20000000,1589,1574,0.7087,0.1888,0.0267,0.000533,2.23,4.33,2012-01-22,9rLZ1ke,"entry_B_7 = ts_rank( ts_std_dev(oth460_3l_vlc,...",PASS,FAIL,PASS,PASS,PASS,PASS,PASS
97,18718474,20000000,1626,1612,0.5838,0.1808,0.0374,0.000619,2.21,3.98,2012-01-22,Y0GLWbo,entry_B_8 = ts_rank( ts_std_dev(oth460_veia_l2...,PASS,PASS,PASS,PASS,PASS,PASS,PASS
98,18063914,20000000,1624,1612,0.5664,0.1744,0.0362,0.000616,2.19,3.95,2012-01-22,MQoMK2k,entry_B_8 = ts_rank( ts_std_dev(oth460_veia_l2...,PASS,PASS,PASS,PASS,PASS,PASS,PASS


In [15]:
result_st1.to_csv('B&Sep13GLB_mixing_add.csv')

In [16]:
#to take a look at the combined result of all new alphas

is_tests_df = hf.concat_is_tests(result)
is_tests_df.head()

,alpha_id,endDate,limit,name,result,startDate,themes,value,year
0,QVLPPEQ,NaN,1.58,LOW_SHARPE,PASS,NaN,NaN,4.9200,NaN
1,QVLPPEQ,NaN,1.00,LOW_FITNESS,PASS,NaN,NaN,2.6700,NaN
2,QVLPPEQ,NaN,0.01,LOW_TURNOVER,PASS,NaN,NaN,0.6534,NaN
3,QVLPPEQ,NaN,0.70,HIGH_TURNOVER,PASS,NaN,NaN,0.6534,NaN
4,QVLPPEQ,NaN,NaN,CONCENTRATED_WEIGHT,PASS,NaN,NaN,NaN,NaN


In [17]:
#making a list of failed alphas
failed_alphas = is_tests_df.query('result=="FAIL"')['alpha_id'].unique()

#making a list of passed alphas
passed_alphas = list(set(is_tests_df['alpha_id']).difference(failed_alphas))

print(f'Failed alphas:{failed_alphas}\nPassed alphas:{passed_alphas}')

Failed alphas:['xk0AAxm' '9rLZ1ke']
Passed alphas:['mbMw3A5', 'zm1K9V8', '5Oo8gzX', 'qloMX5E', 'olMmLRk', 'n2MpVew', 'gMEkjlJ', 'QVLPjQW', '1dQY8Xm', 'JOZjWmj', '1dQYYgJ', '5Oo85N1', '0E0mk31', '9rLZ1Oo', 'R8nRbz0', 'olMmJk2', 'gMEJ3XO', 'zm1qjOO', 'gMEJg50', 'K9jPPbp', 'APM11wd', '9rLZqv2', '69vm835', 'n2MxJpE', 'K9jP5YN', 'plAwOxg', 'bq5GWjK', 'e0o7YQM', 'L1ALZa2', 'n2MpWm8', 'L1AVawa', '5OonRNN', 'ZneYYon', 'gMEJZpm', 'JOZjAax', 'MQoMwl8', 'vl3Rv8z', 'd0A53gg', '5Oo8or5', 'L1AL6Kv', '8Qd5Q9X', 'OZEQq8J', 'QVLPLNG', 'bq5GkVN', 'WG61KzN', 'bq5lbr6', 'k02q2Wz', 'llbeQMe', 'bq5GlXq', '2Lbr6QN', 'L1ALZ96', 'zm1KaaO', 'K9jl7Mz', 'R8nR2ez', 'j0b2lR5', '0E0mmNK', 'QVLZaxK', '3RZqebX', 'bq5GV8R', 'MQoMK2k', 'e0oknzJ', 'olMmYx5', 'MQoPEn8', 'GLp1nGJ', 'K9jlnOl', 'k02qw6K', 'ZneYva0', '71E7weZ', '3RZQ3XP', 'EEX5z0m', '0E0mr0p', 'GLprrz5', '69vXwnJ', 'bq5lObl', 'xk0Aw6m', 'GLpr2OJ', 'Y0GLWbo', 'e0ok7pJ', 'VPgkmR5', 'j0bz9Oe', 'vl3Rg6G', 'olM8qjn', '2LbrRA5', 'Xn5gLK5', 'Xn5mXeb', 'e0okplN', '3R

In [18]:
#calling submit_alpha on all alphas that have passed the submission tests
for alpha_id in passed_alphas:
    hf.set_alpha_properties(s, alpha_id, tags = ['Sep16_GLB_mixing'])

submit tmr!

In [7]:
passed_alphas = ['mbMw3A5', 'zm1K9V8', '5Oo8gzX', 'qloMX5E', 'olMmLRk', 'n2MpVew', 'gMEkjlJ', 'QVLPjQW', '1dQY8Xm', 'JOZjWmj', '1dQYYgJ', '5Oo85N1', '0E0mk31', '9rLZ1Oo', 'R8nRbz0', 'olMmJk2', 'gMEJ3XO', 'zm1qjOO', 'gMEJg50', 'K9jPPbp', 'APM11wd', '9rLZqv2', '69vm835', 'n2MxJpE', 'K9jP5YN', 'plAwOxg', 'bq5GWjK', 'e0o7YQM', 'L1ALZa2', 'n2MpWm8', 'L1AVawa', '5OonRNN', 'ZneYYon', 'gMEJZpm', 'JOZjAax', 'MQoMwl8', 'vl3Rv8z', 'd0A53gg', '5Oo8or5', 'L1AL6Kv', '8Qd5Q9X', 'OZEQq8J', 'QVLPLNG', 'bq5GkVN', 'WG61KzN', 'bq5lbr6', 'k02q2Wz', 'llbeQMe', 'bq5GlXq', '2Lbr6QN', 'L1ALZ96', 'zm1KaaO', 'K9jl7Mz', 'R8nR2ez', 'j0b2lR5', '0E0mmNK', 'QVLZaxK', '3RZqebX', 'bq5GV8R', 'MQoMK2k', 'e0oknzJ', 'olMmYx5', 'MQoPEn8', 'GLp1nGJ', 'K9jlnOl', 'k02qw6K', 'ZneYva0', '71E7weZ', '3RZQ3XP', 'EEX5z0m', '0E0mr0p', 'GLprrz5', '69vXwnJ', 'bq5lObl', 'xk0Aw6m', 'GLpr2OJ', 'Y0GLWbo', 'e0ok7pJ', 'VPgkmR5', 'j0bz9Oe', 'vl3Rg6G', 'olM8qjn', '2LbrRA5', 'Xn5gLK5', 'Xn5mXeb', 'e0okplN', '3RZqeMQ', 'j0b2KYO', 'mbMwe7W', '71E8Q38', 'llbeQA2', 'xk0AA2J', '9rLZvq9', 'APM18VR', 'MQoM51n', 'olM83X5', 'QVLPPEQ', 'qloMAKK']

In [12]:
submit_result = {alpha_id: ace.submit_alpha(s, alpha_id) for alpha_id in passed_alphas[:6]}
submit_result

{'mbMw3A5': False,
 'zm1K9V8': False,
 '5Oo8gzX': False,
 'qloMX5E': False,
 'olMmLRk': False,
 'n2MpVew': False}

### Library Fuctions.

following are some other functions that you can use for your own analysis

**get_alpha_pnl(s, alpha_id)** - to get the pnl for an alpha

**get_alpha_yearly_stats(s, alpha_id)** - to get yearly statistics for an alpha

**get_self_corr(s, alpha_id)** - to get self correlation results for an alpha

**get_prod_corr(s, alpha_id)** - to get prod correlation results for an alpha

**get_check_submission(s, alpha_id)** - to get check submission result for an alpha

**check_self_corr_test(s, alpha_id)** - to check if alpha passes self correlation test (self_corr<0.7)

**check_prod_corr_test(s, alpha_id)** - to check if alpha passes prod correlation test (prod_corr<0.7)

**perfomance_comparison(s, alpha_id)** - to get the result of performance comparison for an alpha merged performance

In [37]:
ace.get_alpha_pnl(s, result[0]['alpha_id'])

,Pnl,alpha_id
Date,,
2012-01-23,0.0,dx9ALqJ
2012-01-24,52190.0,dx9ALqJ
2012-01-25,32038.0,dx9ALqJ
2012-01-26,37403.0,dx9ALqJ
2012-01-27,53631.0,dx9ALqJ
...,...,...
2022-01-17,12935472.0,dx9ALqJ
2022-01-18,12935661.0,dx9ALqJ
2022-01-19,12967474.0,dx9ALqJ
